# Python Advanced: Async, OOP, Decorators, Logging & Testing

**Goal:** Master the Python skills you'll need every day when working with LLM training and inference code.

| Section | Topic |
|---------|-------|
| 1 | async/await and asyncio.gather |
| 2 | Class inheritance: ABC, super(), MRO |
| 3 | Decorators: timing, retry with functools.wraps |
| 4 | Logging: handlers, formatters, best practices |
| 5 | Pytest: fixtures, parametrize, mock |
| 6 | Final exercise: async model loader with retry |

In [ ]:
# Install dependencies (run once)
# !pip install pytest pytest-asyncio pytest-mock

## 1. Async/Await Basics

LLM inference servers handle many concurrent requests. Async IO is the backbone of frameworks like FastAPI, vLLM, and TGI. Let's start from first principles.

In [ ]:
import asyncio
import time

# ------------------------------------------------------------
# Bare-minimum coroutine
# ------------------------------------------------------------
async def fetch_url(url: str, delay: float = 1.0) -> str:
    """Simulate an async HTTP request."""
    print(f"  [start] {url}")
    await asyncio.sleep(delay)          # non-blocking sleep
    print(f"  [done]  {url}")
    return f"<html>{url}</html>"


async def main_sequential():
    """Await one at a time -- SLOW."""
    t0 = time.perf_counter()
    r1 = await fetch_url("https://api.openai.com/v1/models", 1.0)
    r2 = await fetch_url("https://api.openai.com/v1/chat", 1.0)
    r3 = await fetch_url("https://api.openai.com/v1/embeddings", 1.0)
    print(f"Sequential: {time.perf_counter() - t0:.2f}s\n")


async def main_concurrent():
    """Run them all at once with asyncio.gather."""
    t0 = time.perf_counter()
    results = await asyncio.gather(
        fetch_url("https://api.openai.com/v1/models", 1.0),
        fetch_url("https://api.openai.com/v1/chat", 1.0),
        fetch_url("https://api.openai.com/v1/embeddings", 1.0),
    )
    print(f"Concurrent: {time.perf_counter() - t0:.2f}s")
    print(f"Results: {results}")


# Run in a notebook cell
await main_sequential()
await main_concurrent()

In [ ]:
# ------------------------------------------------------------
# Exception handling with gather
# ------------------------------------------------------------
async def risky_fetch(url: str) -> str:
    if "bad" in url:
        raise ValueError(f"Bad URL: {url}")
    await asyncio.sleep(0.5)
    return f"OK: {url}"


async def demo_gather_exceptions():
    # return_exceptions=True collects exceptions instead of raising
    results = await asyncio.gather(
        risky_fetch("/good/a"),
        risky_fetch("/bad/b"),
        risky_fetch("/good/c"),
        return_exceptions=True,
    )
    for i, r in enumerate(results):
        if isinstance(r, Exception):
            print(f"Task {i} FAILED: {r}")
        else:
            print(f"Task {i} OK: {r}")

await demo_gather_exceptions()

In [ ]:
# ------------------------------------------------------------
# TODO: Exercise -- async producer-consumer with asyncio.Queue
# ------------------------------------------------------------
# Simulate token streaming: a producer puts tokens into a queue,
# a consumer reads them and prints them. Fill in the blanks.

async def producer(queue: asyncio.Queue, tokens: list[str]):
    for token in tokens:
        await queue.put(token)
        await asyncio.sleep(0.1)  # simulate generation latency
    await queue.put(None)  # sentinel to signal done

async def consumer(queue: asyncio.Queue):
    # TODO: read from the queue until None is received; print each token
    pass

# TODO: kick off both with asyncio.gather and run them
# tokens = ["Hello", ", ", "world", "!"]
# q = asyncio.Queue()
# await asyncio.gather(producer(q, tokens), consumer(q))

## 2. Class Inheritance: ABC, super(), MRO

LLM codebases are full of abstract base classes -- model registries, dataset interfaces, trainer callbacks. You need to be fluent with ABC and Python's method resolution order.

In [ ]:
from abc import ABC, abstractmethod

# ------------------------------------------------------------
# Abstract base class for any LLM backend
# ------------------------------------------------------------
class LLMBackend(ABC):
    """Every backend must implement generate and count_tokens."""

    def __init__(self, model_name: str):
        self.model_name = model_name

    @abstractmethod
    async def generate(self, prompt: str, max_tokens: int = 256) -> str:
        ...

    @abstractmethod
    def count_tokens(self, text: str) -> int:
        ...

    def __repr__(self):
        return f"{self.__class__.__name__}(model={self.model_name!r})"


# Concrete implementations
class OpenAIBackend(LLMBackend):
    def __init__(self, model_name: str, api_key: str):
        super().__init__(model_name)          # <-- super() calls LLMBackend.__init__
        self.api_key = api_key

    async def generate(self, prompt: str, max_tokens: int = 256) -> str:
        # In real code: await openai.ChatCompletion.acreate(...)
        return f"[OpenAI/{self.model_name}] generated response"

    def count_tokens(self, text: str) -> int:
        return len(text.split())  # placeholder; real code uses tiktoken


class LocalBackend(LLMBackend):
    def __init__(self, model_name: str, device: str = "cuda"):
        super().__init__(model_name)
        self.device = device

    async def generate(self, prompt: str, max_tokens: int = 256) -> str:
        return f"[Local/{self.model_name}:{self.device}] generated response"

    def count_tokens(self, text: str) -> int:
        return len(text.split())


# Cannot instantiate ABC directly:
# backend = LLMBackend("gpt-4")  # TypeError!

openai_be = OpenAIBackend("gpt-4o", api_key="sk-...")
local_be  = LocalBackend("qwen2.5-7b", device="cuda:0")
print(openai_be, local_be, sep="\n")

In [ ]:
# ------------------------------------------------------------
# MRO (Method Resolution Order) -- critical for mixins
# ------------------------------------------------------------

class LoggingMixin:
    """Mixin that adds structured logging to any backend."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)  # delegates to next class in MRO!
        self.call_count = 0

    async def generate(self, prompt: str, max_tokens: int = 256) -> str:
        self.call_count += 1
        print(f"[LOG] generate call #{self.call_count} (tokens={max_tokens})")
        return await super().generate(prompt, max_tokens)


class CachingMixin:
    """Mixin that caches generate results."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cache = {}

    async def generate(self, prompt: str, max_tokens: int = 256) -> str:
        key = (prompt, max_tokens)
        if key not in self._cache:
            self._cache[key] = await super().generate(prompt, max_tokens)
        else:
            print("[CACHE] hit!")
        return self._cache[key]


# Compose a backend with logging + caching
class SmartBackend(LoggingMixin, CachingMixin, LocalBackend):
    pass

# Check the MRO
print("MRO:")
for cls in SmartBackend.__mro__:
    print(f"  {cls}")

be = SmartBackend("qwen2.5-7b")
print(f"\n{await be.generate('Hello')}")
print(f"{await be.generate('Hello')}")   # should hit cache

In [ ]:
# ------------------------------------------------------------
# TODO: Exercise
# ------------------------------------------------------------
# Create a RateLimitedBackend mixin that enforces max 2 calls/sec
# using asyncio.sleep. Stack it with LocalBackend.
#
# class RateLimitedMixin:
#     ...  # your code here
#
# class ThrottledBackend(RateLimitedMixin, LocalBackend):
#     pass

## 3. Decorators: Timing & Retry

Decorators are everywhere in LLM training: `@torch.no_grad()`, `@contextmanager` for device placement, custom retry wrappers, and profiling hooks.

In [ ]:
import functools
import time
import random

# ------------------------------------------------------------
# 3a. Timing decorator
# ------------------------------------------------------------
def timer(func):
    """Print the runtime of the decorated function."""
    @functools.wraps(func)       # preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f"[{func.__name__}] {elapsed:.4f}s")
        return result
    return wrapper

@timer
def compute_embeddings(texts: list[str]) -> list[float]:
    time.sleep(0.3)
    return [len(t) * 0.01 for t in texts]

vecs = compute_embeddings(["hello", "world"])
print(f"Result: {vecs}")
print(f"Function name preserved: {compute_embeddings.__name__}")

In [ ]:
# ------------------------------------------------------------
# 3b. Retry decorator (essential for API calls)
# ------------------------------------------------------------
def retry(max_attempts: int = 3, delay: float = 1.0, backoff: float = 2.0,
          exceptions: tuple = (Exception,)):
    """Retry a function with exponential backoff.

    Args:
        max_attempts: Maximum number of attempts (including the first).
        delay: Initial delay between retries in seconds.
        backoff: Multiplier applied to delay after each failure.
        exceptions: Tuple of exception types to catch.
    """
    def decorator(func):
        @functools.wraps(func)
        async def async_wrapper(*args, **kwargs):
            last_exc = None
            current_delay = delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return await func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    if attempt == max_attempts:
                        raise
                    print(f"[retry] attempt {attempt}/{max_attempts} failed: {e}. "
                          f"Retrying in {current_delay:.1f}s...")
                    await asyncio.sleep(current_delay)
                    current_delay *= backoff
            raise last_exc  # type: ignore

        @functools.wraps(func)
        def sync_wrapper(*args, **kwargs):
            last_exc = None
            current_delay = delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    if attempt == max_attempts:
                        raise
                    print(f"[retry] attempt {attempt}/{max_attempts} failed: {e}. "
                          f"Retrying in {current_delay:.1f}s...")
                    time.sleep(current_delay)
                    current_delay *= backoff
            raise last_exc  # type: ignore

        if asyncio.iscoroutinefunction(func):
            return async_wrapper
        return sync_wrapper
    return decorator


# Demonstrate with an unreliable API call
call_count = 0

@retry(max_attempts=4, delay=0.3, backoff=1.5, exceptions=(ValueError,))
async def flaky_api_call(prompt: str) -> str:
    global call_count
    call_count += 1
    if call_count < 3:
        raise ValueError(f"Temporary server error (call {call_count})")
    return f"Response to: {prompt}"

print(await flaky_api_call("What is the capital of France?"))

In [ ]:
# ------------------------------------------------------------
# TODO: Exercise -- write an @audit_log decorator
# ------------------------------------------------------------
# It should:
#  - Log the function name, args, kwargs, and return value at DEBUG level
#  - Log any exception at ERROR level and re-raise
#  - Preserve the function signature with functools.wraps
#
# import logging
# logger = logging.getLogger(__name__)
# def audit_log(func):
#     ...  # your code
    

## 4. Logging Module: Handlers & Formatters

`print()` does not scale to production. The `logging` module gives you levels, structured output, and routing.

In [ ]:
import logging
import sys

# ------------------------------------------------------------
# 4a. Basic logging setup
# ------------------------------------------------------------

# Always use module-level loggers
logger = logging.getLogger(__name__)

# One-time configuration (usually in your app's entry point)
def setup_logging(level: int = logging.INFO):
    root = logging.getLogger()
    root.setLevel(level)

    # Console handler
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(level)
    console.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s:%(lineno)d | %(message)s",
        datefmt="%H:%M:%S"
    ))
    root.addHandler(console)

    # Suppress noisy third-party loggers
    logging.getLogger("urllib3").setLevel(logging.WARNING)
    logging.getLogger("httpx").setLevel(logging.WARNING)

setup_logging(logging.DEBUG)

logger.debug("Debug -- detailed diagnostic")
logger.info("Info -- model loaded successfully")
logger.warning("Warning -- low GPU memory")
logger.error("Error -- request timeout")
logger.critical("Critical -- CUDA OOM")

In [ ]:
# ------------------------------------------------------------
# 4b. File handler + JSON formatter for structured logs
# ------------------------------------------------------------
import json
from datetime import datetime
from pathlib import Path

class JSONFormatter(logging.Formatter):
    """Emit log records as JSON lines -- great for log aggregation."""
    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "ts":       datetime.utcnow().isoformat() + "Z",
            "level":    record.levelname,
            "logger":   record.name,
            "msg":      record.getMessage(),
            "module":   record.module,
            "line":     record.lineno,
        }
        if record.exc_info and record.exc_info[1]:
            payload["exception"] = str(record.exc_info[1])
        return json.dumps(payload, ensure_ascii=False)


# Set up a file logger with JSON output
json_handler = logging.FileHandler("/tmp/llm_app.jsonl")
json_handler.setLevel(logging.DEBUG)
json_handler.setFormatter(JSONFormatter())

file_logger = logging.getLogger("llm.inference")
file_logger.addHandler(json_handler)
file_logger.propagate = False  # don't duplicate to root handlers

file_logger.info("Model loaded", extra={"model": "qwen2.5-7b", "device": "cuda:0"})
try:
    1 / 0
except ZeroDivisionError:
    file_logger.exception("Math error during preprocessing")

print("Written to /tmp/llm_app.jsonl:")
print(Path("/tmp/llm_app.jsonl").read_text())

In [ ]:
# ------------------------------------------------------------
# TODO: Exercise -- context manager for timing blocks
# ------------------------------------------------------------
# Write a context manager (or decorator) that logs the duration
# of a code block at INFO level. Use it to wrap a model inference call.
#
# @contextmanager
# def log_duration(operation_name: str):
#     ...  # your code

## 5. Pytest: Fixtures, Parametrize, Mock

Testing async code, mocking external APIs, and parametric tests are daily skills.

In [ ]:
# ------------------------------------------------------------
# 5a. Fixtures -- reusable test dependencies
# ------------------------------------------------------------
# (This cell is a pytest demo. Run with: pytest --tb=short this_notebook.py)

import pytest

# Fixtures can be async too (requires pytest-asyncio)
@pytest.fixture
def sample_prompts():
    """Reusable test data."""
    return [
        {"role": "user", "content": "Hello"},
        {"role": "user", "content": "What is 2+2?"},
        {"role": "user", "content": "Write a haiku about GPUs"},
    ]

@pytest.fixture
def mock_tokenizer():
    """A fake tokenizer for unit tests."""
    class FakeTokenizer:
        def encode(self, text: str) -> list[int]:
            return [ord(c) for c in text]
        def decode(self, ids: list[int]) -> str:
            return "".join(chr(i) for i in ids)
    return FakeTokenizer()


def test_fixtures_work(sample_prompts, mock_tokenizer):
    """This test uses both fixtures."""
    assert len(sample_prompts) == 3
    encoded = mock_tokenizer.encode("Hi")
    assert encoded == [72, 105]
    assert mock_tokenizer.decode(encoded) == "Hi"

# In a notebook we call the test directly (normally pytest collects it)
test_fixtures_work(
    sample_prompts(),
    mock_tokenizer()
)
print("test_fixtures_work PASSED")

In [ ]:
# ------------------------------------------------------------
# 5b. Parametrize -- test many inputs with one function
# ------------------------------------------------------------

@pytest.mark.parametrize("text,expected_tokens", [
    ("hello", 1),
    ("hello world", 2),
    ("hello world, this is a test", 6),
    ("", 0),
])
def test_token_count(text, expected_tokens):
    """Our naive token counter should match expected word count."""
    token_count = len(text.split()) if text.strip() else 0
    assert token_count == expected_tokens


# Manually run the parametrized cases in the notebook
for text, expected in [("hello", 1), ("hello world", 2),
                       ("hello world, this is a test", 6), ("", 0)]:
    test_token_count(text, expected)
print("All parametrized cases PASSED")

In [ ]:
# ------------------------------------------------------------
# 5c. Mocking -- isolate your code from external dependencies
# ------------------------------------------------------------
from unittest.mock import AsyncMock, patch, MagicMock

# Imagine this is the function we want to test
async def summarize_text(client, text: str) -> str:
    """Call an external API to summarize text."""
    response = await client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": f"Summarize: {text}"}],
    )
    return response.choices[0].message.content


async def test_summarize_with_mock():
    """Test summarize_text without hitting the real API."""
    mock_client = MagicMock()
    mock_client.chat.completions.create = AsyncMock(return_value=MagicMock(
        choices=[MagicMock(message=MagicMock(content="A short summary"))]
    ))

    result = await summarize_text(mock_client, "A very long article...")
    assert result == "A short summary"

    # Assert the API was called with correct arguments
    mock_client.chat.completions.create.assert_called_once()
    call_kwargs = mock_client.chat.completions.create.call_args.kwargs
    assert call_kwargs["model"] == "gpt-4o"
    print("test_summarize_with_mock PASSED")

await test_summarize_with_mock()

In [ ]:
# ------------------------------------------------------------
# TODO: Exercise -- write test for the retry decorator
# ------------------------------------------------------------
# Write a test that verifies @retry actually retries on failure.
# Use a mock that fails N times then succeeds.
#
# async def test_retry_on_flaky_function():
#     call_counter = 0
#
#     @retry(max_attempts=3, delay=0.01)
#     async def flaky():
#         nonlocal call_counter
#         call_counter += 1
#         if call_counter < 3:
#             raise ValueError("fail")
#         return "success"
#
#     result = await flaky()
#     assert result == "success"
#     assert call_counter == 3
    

## 6. Final Exercise: Async Model Loader with Retry

Combine everything you have learned:

- Abstract base class for a model loader
- Async loading with asyncio
- @retry decorator for flaky downloads
- Structured logging
- A pytest test for your loader

In [ ]:
# ============================================================
# SOLUTION -- build this yourself first, then compare
# ============================================================

import hashlib
from abc import ABC, abstractmethod
from pathlib import Path

logger = logging.getLogger("model_loader")


class ModelLoader(ABC):
    """Abstract base for downloading and verifying model files."""

    def __init__(self, cache_dir: str = "/tmp/models"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    @abstractmethod
    def model_uri(self) -> str:
        """Return the URI to download from."""
        ...

    @abstractmethod
    def expected_checksum(self) -> str | None:
        """Return expected SHA256 (or None to skip verification)."""
        ...

    @retry(max_attempts=3, delay=0.5, backoff=2.0, exceptions=(IOError, ConnectionError))
    async def download(self) -> Path:
        """Download the model file with retries."""
        dest = self.cache_dir / Path(self.model_uri()).name
        if dest.exists():
            logger.info("Model already cached at %s", dest)
            return dest

        logger.info("Downloading %s -> %s", self.model_uri(), dest)
        # Simulate a download that may fail
        await asyncio.sleep(0.5)
        dest.write_text('{"weights": "fake", "version": 1}')
        logger.info("Download complete: %s", dest)
        return dest

    def verify(self, path: Path) -> bool:
        """Check SHA256 checksum."""
        expected = self.expected_checksum()
        if expected is None:
            logger.warning("No checksum provided -- skipping verification")
            return True
        actual = hashlib.sha256(path.read_bytes()).hexdigest()
        ok = actual == expected
        if not ok:
            logger.error("Checksum mismatch! expected=%s actual=%s", expected, actual)
        return ok

    async def load(self) -> Path:
        """Full pipeline: download + verify."""
        path = await self.download()
        if not self.verify(path):
            raise IOError(f"Checksum verification failed for {path}")
        logger.info("Model ready: %s", path)
        return path


class Qwen25Loader(ModelLoader):
    def model_uri(self) -> str:
        return "https://huggingface.co/Qwen/Qwen2.5-0.5B/resolve/main/model.safetensors"

    def expected_checksum(self) -> str | None:
        return None  # Skip checksum for the demo


# Run it
loader = Qwen25Loader()
path = await loader.load()
print(f"Model loaded at: {path}")

In [ ]:
# ------------------------------------------------------------
# TODO: Write a pytest test for ModelLoader
# ------------------------------------------------------------
# Test that:
#  1. download() creates a file in cache_dir
#  2. A second call to download() returns the cached file (no re-download)
#  3. verify() returns False when checksum does not match
#
# Use tmp_path fixture for an isolated cache directory.
#
# class FakeLoader(ModelLoader):
#     def model_uri(self): return "http://example.com/model.bin"
#     def expected_checksum(self): return "abc123"
#
# async def test_download_caches_file(tmp_path):
#     ...
#
# async def test_verify_detects_mismatch(tmp_path):
#     ...

---

## What You've Built

| Skill | Applied In |
|-------|-----------|
| `asyncio.gather`, `asyncio.Queue` | Concurrent API calls, token streaming |
| ABC, `super()`, MRO, mixins | Model backend abstraction, composable features |
| Decorators with `functools.wraps` | Retry logic, profiling, audit logging |
| `logging` with handlers & JSON formatter | Production observability |
| `pytest` fixtures, parametrize, mock | Isolated, fast unit tests for ML code |

These patterns appear in every major LLM codebase. Keep this notebook as a reference.